# Secure UPI: Machine Learning-Driven Fraud Detection System
## Academic FinTech Research & Model Development Notebook

### Project Objective
Develop an intelligent real-time fraud detection pipeline for Unified Payments Interface (UPI) transactions by evaluating behavioral anomalies, transaction dynamics, device/location parameters, and historical user baselines.

### Machine Learning Models Evaluated:
1. **Logistic Regression** (Linear baseline with balanced class weights)
2. **Random Forest Classifier** (Ensemble tree-based bagging with balanced subsampling)
3. **XGBoost Classifier** (Gradient boosting with scale_pos_weight optimization)

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from xgboost import XGBClassifier

print("All libraries imported successfully.")

## 1. Dataset Loading & Exploratory Data Analysis (EDA)

In [2]:
# Load the dataset
df = pd.read_csv('../backend/ml/dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [3]:
# Summary statistics and class balance
print("--- Class Distribution ---")
print(df['is_fraud'].value_counts())
print(f"\nFraud Rate: {df['is_fraud'].mean()*100:.2f}%")

print("\n--- Feature Summary ---")
df.describe().T[['mean', 'std', 'min', '50%', 'max']]

## 2. Feature Engineering & Preprocessing
We transform raw transaction inputs into fraud-predictive indicators:
- Amount deviation ratio (`amount / average_amount`)
- Unusual hour flag (1 AM - 5 AM)
- High-value spike flag
- Rapid transaction velocity flag
- Device change & Location anomaly flags

In [4]:
from backend.ml.preprocess import UPIFeaturePipeline, ALL_MODEL_INPUT_COLUMNS

X = df[ALL_MODEL_INPUT_COLUMNS]
y = df['is_fraud']

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

pipeline = UPIFeaturePipeline()
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)
feature_names = pipeline.get_feature_names()

print(f"Transformed Feature Count: {len(feature_names)}")
print("Features:", feature_names)

## 3. Model Training & Comparative Evaluation
We train and benchmark Logistic Regression, Random Forest, and XGBoost using comprehensive evaluation metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC).

In [5]:
scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced_subsample', random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42)
}

evaluation_results = []
for name, model in models.items():
    model.fit(X_train_trans, y_train)
    y_pred = model.predict(X_test_trans)
    y_proba = model.predict_proba(X_test_trans)[:, 1]
    
    evaluation_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(evaluation_results)
print("=== Comparative Model Evaluation ===")
results_df

## 4. Feature Importance & Risk Explainability
Examining the most influential features contributing to fraud detection decisions.

In [6]:
best_rf = models['Random Forest']
importances = pd.Series(best_rf.feature_importances_, index=feature_names).sort_values(ascending=False)

print("Top Feature Importances (Random Forest):")
print(importances.head(10))

## 5. Model Serialization & Summary
The best-performing model is serialized with `joblib` for deployment into the real-time Flask API backend.